# Reprocess and draw ISA figures from completed ISA outputs

This notebook starts from completed ISA result CSV files (`motif_single_isa.csv`, `motif_combi_isa.csv`, `null_isa.csv`, `null_interaction.csv`, `motif_locs.csv`, `coop_tf_pair_t*.csv`).

It does **not** rerun model inference, motif ablation or ISA scoring. It only rebuilds clean source tables and draws each manuscript figure step by step.


## Figure contract

- **Input layer:** completed ISA outputs from the mounted Google Drive result directory.
- **Processing layer:** source-table rebuilding with explicit task labels (`CAGE`, `DEV`, `HK`), task-specific null thresholds, `q < 0.1` filtered TF-pair definition, and motif-position overlap classes.
- **Figure layer:** each figure is drawn in a separate cell and saved as PNG/SVG/PDF in the output directory.
- **No new ISA:** this notebook does not call `run_single_isa`, `run_combi_isa`, `run_null_isa`, or `run_null_interaction`.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd
from IPython.display import Image, display, Markdown

PIPELINE_SCRIPT = Path(r"F:\phd\Drophila\3Model_motif_discovering\ISA\result\all_reconstructed_figures_from_gdrive\nature_final_package\reprocess_from_raw_isa_outputs.py")
INPUT_RESULT_ROOT = Path(r"G:\我的云端硬盘\DeepEpromote\Drosophila\DeepCAGE\script\Motif_discover\Ep_ISA_NEW\result")
OUTPUT_DIR = Path(r"F:\phd\Drophila\3Model_motif_discovering\ISA\result\all_reconstructed_figures_from_gdrive\nature_final_package_reprocessed_from_gdrive")
SOURCE_DIR = OUTPUT_DIR / "source_data"

print("Pipeline script:", PIPELINE_SCRIPT)
print("Completed ISA input root:", INPUT_RESULT_ROOT)
print("Figure output dir:", OUTPUT_DIR)
assert PIPELINE_SCRIPT.exists(), PIPELINE_SCRIPT
assert INPUT_RESULT_ROOT.exists(), INPUT_RESULT_ROOT


## 1. Load the completed ISA result files

In [ ]:
TASKS = [
    {"label": "CAGE", "folder": "results_cage", "track": 0},
    {"label": "DEV", "folder": "results_dev", "track": 0},
    {"label": "HK", "folder": "results_hk", "track": 1},
]

required = [
    "motif_locs.csv",
    "motif_single_isa.csv",
    "motif_combi_isa.csv",
    "null_isa.csv",
    "null_interaction.csv",
    "tf_importance.csv",
]

rows = []
for task in TASKS:
    data_dir = INPUT_RESULT_ROOT / task["folder"] / "Data"
    for filename in required:
        path = data_dir / filename
        rows.append({
            "task": task["label"],
            "track": task["track"],
            "file": filename,
            "exists": path.exists(),
            "path": str(path),
            "rows": len(pd.read_csv(path)) if path.exists() else None,
        })

loaded_summary = pd.DataFrame(rows)
display(loaded_summary)
assert loaded_summary["exists"].all()


## 2. Import the reprocessing pipeline

In [ ]:
spec = importlib.util.spec_from_file_location("isa_reprocess", PIPELINE_SCRIPT)
rp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = rp
spec.loader.exec_module(rp)

# Set the pipeline globals for this notebook run.
rp.RESULT_ROOT = INPUT_RESULT_ROOT
rp.OUTPUT = OUTPUT_DIR
rp.SD = SOURCE_DIR

print("Configured pipeline:")
print("  RESULT_ROOT =", rp.RESULT_ROOT)
print("  OUTPUT      =", rp.OUTPUT)
print("  SOURCE_DIR  =", rp.SD)
print("  q threshold =", rp.Q_VALUE_THRESHOLD)
print("  null percentile =", rp.NULL_PERCENTILE)


## 3. Rebuild source tables from completed ISA outputs

In [ ]:
# This rebuilds source_data from completed ISA CSVs and copies plotting scripts into OUTPUT_DIR.
# It does not render figures yet.
rp.write_all_sources()

display(Markdown((OUTPUT_DIR / "REPROCESSING_QA_REPORT.md").read_text(encoding="utf-8")))
display(pd.read_json(OUTPUT_DIR / "analysis_config.json"))


## 4. Load plotting code from the rebuilt output package

In [ ]:
REDRAW_SCRIPT = OUTPUT_DIR / "redraw_final_figures_from_source_data.py"
FORMULA_SCRIPT = OUTPUT_DIR / "draw_isa_formula_schematic.py"

spec = importlib.util.spec_from_file_location("redraw_figures", REDRAW_SCRIPT)
draw = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = draw
spec.loader.exec_module(draw)
draw.setup()

spec_formula = importlib.util.spec_from_file_location("formula_fig", FORMULA_SCRIPT)
formula = importlib.util.module_from_spec(spec_formula)
sys.modules[spec_formula.name] = formula
spec_formula.loader.exec_module(formula)

print("Loaded plotting code from:", REDRAW_SCRIPT)


## 5. Draw ISA formula schematic

In [ ]:
formula.draw()
display(Image(filename=str(OUTPUT_DIR / "ISA_formula_schematic_latest.png")))


## 6. Draw Fig. 1 | Full-task ISA landscape

In [ ]:
display(pd.read_csv(SOURCE_DIR / "fig0_global_task_null_comparison_nature_skill_source_data.csv"))
draw.fig1()
display(Image(filename=str(OUTPUT_DIR / "Fig1_full_task_global_view_latest.png")))


## 7. Draw Fig. 2 | Real-vs-null distributions

In [ ]:
display(pd.read_csv(SOURCE_DIR / "fig2_real_vs_null_nature_skill_source_data.csv"))
draw.fig2()
display(Image(filename=str(OUTPUT_DIR / "Fig2_full_task_real_vs_null_latest.png")))


## 8. Draw Fig. 3 | Pair-interaction sign structure

In [ ]:
display(Markdown("### All pair interactions, null-binned"))
display(pd.read_csv(SOURCE_DIR / "fig3_all_instance_null_classes_source_data.csv"))
display(Markdown("### Filtered TF-pair sign counts"))
display(pd.read_csv(SOURCE_DIR / "fig3_coop_ecdf_sign_nature_skill_source_sign_counts.csv"))
draw.fig3()
display(Image(filename=str(OUTPUT_DIR / "Fig3_full_task_pair_sign_latest.png")))


## 9. Draw Fig. 4 | Fixed-flank distance-context control

In [ ]:
fig4_source = pd.read_csv(SOURCE_DIR / "fig4_fixed_flank_context_source_data.csv")
display(fig4_source)
draw.fig4()
display(Image(filename=str(OUTPUT_DIR / "Fig4_full_task_distance_context_latest.png")))


## 10. Draw Fig. 5 | Motif-position overlap classes

In [ ]:
classes = pd.read_csv(SOURCE_DIR / "fig1_sequence_overview_nature_skill_source_sequence_classes.csv")
reuse = pd.read_csv(SOURCE_DIR / "fig1_sequence_overview_nature_skill_source_position_reuse.csv")
display(Markdown("### Motif-position overlap class counts"))
display(classes["class"].value_counts().rename_axis("class").reset_index(name="n"))
display(Markdown("### Position-reuse support"))
display(reuse)
draw.fig5()
display(Image(filename=str(OUTPUT_DIR / "Fig5_cage_associated_overlap_classes_latest.png")))


## 11. Draw Fig. 6 | Recurrent CAGE-associated pair shifts

In [ ]:
pairs = pd.read_csv(SOURCE_DIR / "fig5_recurrent_combined_nature_skill_source_pairs.csv")
deltas = pd.read_csv(SOURCE_DIR / "fig5_recurrent_combined_nature_skill_source_paired_differences.csv")
display(Markdown("### Recurrent pair rows"))
display(pairs)
display(Markdown("### Paired differences"))
display(deltas)
draw.fig6()
display(Image(filename=str(OUTPUT_DIR / "Fig6_cage_associated_recurrent_pair_shift_latest.png")))


## 12. Draw Extended Data Fig. 1-5

In [ ]:
extended = [
    ("Extended Data Fig. 1", draw.ext1, "ExtDataFig1_full_task_pair_score_matrices_latest.png", "ed1_full_task_pair_score_matrices_source_data.csv"),
    ("Extended Data Fig. 2", draw.ext2, "ExtDataFig2_full_task_pair_quality_metrics_latest.png", "ed2_full_task_pair_quality_metrics_source_data.csv"),
    ("Extended Data Fig. 3", draw.ext3, "ExtDataFig3_legacy_category_sign_audit_latest.png", "ed3_legacy_category_sign_audit_source_data.csv"),
    ("Extended Data Fig. 4", draw.ext4, "ExtDataFig4_motif_position_architecture_latest.png", "ed4_full_task_motif_position_architecture_source_data.csv"),
    ("Extended Data Fig. 5", draw.ext5, "ExtDataFig5_fixed_flank_distance_support_latest.png", "ed5_fixed_flank_distance_support_source_data.csv"),
]

for title, fn, image_name, source_name in extended:
    display(Markdown(f"### {title}"))
    source = pd.read_csv(SOURCE_DIR / source_name)
    display(source.head(20))
    fn()
    display(Image(filename=str(OUTPUT_DIR / image_name)))


## 13. Final QA checks

In [ ]:
# Check labels, thresholds and expected output images.
display(Markdown((OUTPUT_DIR / "REPROCESSING_QA_REPORT.md").read_text(encoding="utf-8")))

expected = [
    "ISA_formula_schematic_latest.png",
    "Fig1_full_task_global_view_latest.png",
    "Fig2_full_task_real_vs_null_latest.png",
    "Fig3_full_task_pair_sign_latest.png",
    "Fig4_full_task_distance_context_latest.png",
    "Fig5_cage_associated_overlap_classes_latest.png",
    "Fig6_cage_associated_recurrent_pair_shift_latest.png",
    "ExtDataFig1_full_task_pair_score_matrices_latest.png",
    "ExtDataFig2_full_task_pair_quality_metrics_latest.png",
    "ExtDataFig3_legacy_category_sign_audit_latest.png",
    "ExtDataFig4_motif_position_architecture_latest.png",
    "ExtDataFig5_fixed_flank_distance_support_latest.png",
]
qa = pd.DataFrame({"file": expected, "exists": [(OUTPUT_DIR / x).exists() for x in expected]})
display(qa)
assert qa["exists"].all()
